# 🚀 RAGForge: Two-Stage Alignment Pipeline (Cold-Start SFT + GRPO Reinforcement Learning)

This notebook trains a compact, high-reasoning language model (**`Qwen/Qwen2.5-1.5B-Instruct`**) using the **Two-Stage DeepSeek-R1 Alignment Recipe** grounded on real-world technical RAG community discourse.

---

### 🔬 The Two-Stage Architecture & Upgrades:
1. **Stage 1 — Cold-Start SFT (Domain Knowledge Acquisition & Dynamic CoT):**
   - Pure RL cannot reward what the model has never seen. In Cold-Start SFT, the model trains on curated technical discussions.
   - **Dynamic Chain-of-Thought:** Instead of static boilerplate, each sample features a problem-specific `<think>` trace analyzing diagnostic mechanisms, failure modes (lexical mismatch, embedding dilution, lost in the middle), and target architectural solutions.
   - **Answer Hygiene:** Automatically strips forum chatter ("Hope this helps!", hallucinated URLs) and filters for substantive, high-depth technical solutions (>=150 chars).

2. **Stage 2 — GRPO Reinforcement Learning (Reasoning Policy Refinement):**
   - Group Sampling ($G=4$): Model generates 4 candidate reasoning paths per prompt.
   - Multi-Objective Rule Rewards:
     - `reward_reasoning_format`: Strict verification of `<think>` and `<answer>` tags.
     - `reward_rag_concept_density`: Scores domain mechanisms in `<answer>` AND awards a **CoT Planning Bonus** for mechanisms brainstormed in `<think>`!
     - `reward_anti_repetition`: Penalizes empty responses, 4-gram repetition loops, hallucinated URLs, and forum noise.

3. **Stage 3 — Deep Diagnostic Evaluation & Full Debugging Logging:**
   - Evaluates on 13 held-out canonical test cases (`rag_eval.jsonl`) with `max_new_tokens=512` (zero mid-sentence truncation!).
   - Logs token throughput, concept recall %, thinking adoption, and saves all raw answers, extracted thinking traces, matched concepts, and **missing concepts (where it fumbled!)** to `post_grpo_eval_report.json`.

4. **Stage 4 — Hugging Face Hub Publishing & Local Backup:**
   - Pushes LoRA adapter and merged standalone model to Hugging Face Hub and provides local zip download.

### Step 1: Check GPU Acceleration
Ensure your Colab runtime is set to **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies & Fix Colab Incompatibilities
Uninstalls incompatible pre-installed `torchao` and applies patches to eliminate Colab C++ ABI collisions.

In [ ]:
# 1. Uninstall outdated Colab torchao (eliminates version collision with peft)
!pip uninstall -y -q torchao

# 2. Install modern HuggingFace TRL, PEFT, Transformers, Datasets, and Polars
!pip install --upgrade -q "transformers>=4.48.0" "trl>=0.12.0" "peft>=0.13.0" accelerate polars pyarrow datasets

# 3. Neutralize Colab torchvision & torchao ABI version collisions
import transformers.utils.import_utils as t_utils
t_utils.is_torchvision_available = lambda: False
import peft.import_utils as p_utils
p_utils.is_torchao_available = lambda: False
try:
    import peft.utils.quantization_utils as q_utils
    q_utils.is_torchao_available = lambda: False
except Exception:
    pass

# 4. CUDA Memory allocator settings to prevent fragmentation on Tesla T4 (15 GB)
import os
import torch
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("✅ Dependencies loaded and Colab runtime patches applied!")

### Step 3: Dataset Ingestion & Auto-Setup
Upload `grpo_train_dataset.parquet` and automatically initialize the canonical 13-question `rag_eval.jsonl` benchmark so it never encounters a `FileNotFoundError`.

In [ ]:
import os
import json
from google.colab import files

# 1. Upload or verify grpo_train_dataset.parquet
if not os.path.exists("grpo_train_dataset.parquet"):
    print("Please upload your data/processed/grpo_train_dataset.parquet:")
    uploaded = files.upload()
else:
    print("✅ grpo_train_dataset.parquet already present!")

# 2. Embedded canonical 13-question benchmark (guarantees rag_eval.jsonl is always present)
RAG_EVAL_BENCHMARK = [
  {
    "id": "rag_ret_001",
    "category": "retrieval",
    "question": "A production RAG system retrieves documents using dense cosine similarity with OpenAI text-embedding-3-small. Users report that queries containing specific part numbers (e.g., 'SKU-8921-X') consistently return irrelevant general catalog pages instead of the exact product specifications. Why does this happen, and what is the standard architectural fix?",
    "expected_concepts": ["dense embeddings", "out-of-vocabulary tokens", "lexical search", "BM25 / sparse retrieval", "hybrid search", "Reciprocal Rank Fusion (RRF)"]
  },
  {
    "id": "rag_ret_002",
    "category": "retrieval",
    "question": "Compare bi-encoder dense retrieval (e.g., bge-large) against cross-encoder reranking (e.g., bge-reranker-large). Why can't we simply use a cross-encoder across the entire document collection of 10 million passages?",
    "expected_concepts": ["bi-encoder", "cross-encoder", "computational complexity", "cross-attention", "offline indexing", "two-stage retrieval"]
  },
  {
    "id": "rag_ret_003",
    "category": "retrieval",
    "question": "What is Hypothetical Document Embeddings (HyDE), and in what specific scenario does HyDE degrade retrieval performance instead of improving it?",
    "expected_concepts": ["HyDE", "zero-shot generation", "hallucination propagation", "domain-specific terminology", "semantic drift"]
  },
  {
    "id": "rag_idx_001",
    "category": "chunking_and_indexing",
    "question": "A developer splits documents into 2048-token chunks to ensure the LLM has maximum context. However, retrieval accuracy plummets compared to using 256-token chunks. Explain the technical phenomenon behind this degradation in embedding search.",
    "expected_concepts": ["embedding dilution", "semantic pooling", "vector representational capacity", "granularity mismatch", "chunk size trade-off"]
  },
  {
    "id": "rag_idx_002",
    "category": "chunking_and_indexing",
    "question": "In an HNSW (Hierarchical Navigable Small World) vector index, what do the hyperparameters 'M', 'efConstruction', and 'efSearch' control, and what is the trade-off of increasing efSearch at query time?",
    "expected_concepts": ["HNSW graph", "M (bidirectional links)", "efConstruction", "efSearch", "recall vs latency", "priority queue size"]
  },
  {
    "id": "rag_idx_003",
    "category": "chunking_and_indexing",
    "question": "Explain the difference between pre-filtering and post-filtering when applying metadata filters in a vector database (e.g., filtering by tenant_id = 'org_42'). Why can naive post-filtering cause a query to return zero results even when matching documents exist?",
    "expected_concepts": ["pre-filtering", "post-filtering", "top-k cutoff", "tenant isolation", "vector payload filtering", "HNSW graph traversal"]
  },
  {
    "id": "rag_gen_001",
    "category": "generation_and_grounding",
    "question": "What is the 'Lost in the Middle' phenomenon in transformer context windows, and how should retrieved RAG chunks be positioned in the prompt to maximize generation accuracy?",
    "expected_concepts": ["Lost in the Middle", "primacy bias", "recency bias", "chunk ordering", "attention degradation"]
  },
  {
    "id": "rag_gen_002",
    "category": "generation_and_grounding",
    "question": "How can a system design engineer enforce strict citation attribution and prevent hallucinated references in a RAG response? Describe at least two independent validation layers beyond standard system prompting.",
    "expected_concepts": ["citation attribution", "verbatim quote extraction", "string matching / fuzzy verification", "nli / entailment model", "hallucination mitigation"]
  },
  {
    "id": "rag_eval_001",
    "category": "evaluation_and_benchmarks",
    "question": "In RAG evaluation frameworks (such as Ragas or TruLens), distinguish between 'Faithfulness' (or Groundedness) and 'Answer Relevance'. Can a response have 100% Faithfulness but 0% Answer Relevance? Give a concrete example.",
    "expected_concepts": ["faithfulness / groundedness", "answer relevance", "retrieved context adherence", "query addressing", "independent metrics"]
  },
  {
    "id": "rag_eval_002",
    "category": "evaluation_and_benchmarks",
    "question": "When using an LLM-as-a-judge to evaluate RAG answer quality, what are the three most prevalent systematic biases, and what calibration techniques neutralize them?",
    "expected_concepts": ["LLM-as-a-judge", "verbosity bias", "position bias", "self-enhancement bias", "swap evaluation", "reference-guided rubrics"]
  },
  {
    "id": "rag_adv_001",
    "category": "agentic_and_graph_rag",
    "question": "What is Corrective RAG (CRAG), and how does its retrieval confidence evaluator dynamically route between document refinement and external search?",
    "expected_concepts": ["Corrective RAG (CRAG)", "retrieval evaluator", "confidence threshold", "correct / incorrect / ambiguous", "external web search fallback", "strip irrelevant sentences"]
  },
  {
    "id": "rag_adv_002",
    "category": "agentic_and_graph_rag",
    "question": "In what specific problem domain does GraphRAG (knowledge graph-based RAG) fundamentally outperform chunk-based vector search, and what are the primary engineering bottlenecks in building a GraphRAG index?",
    "expected_concepts": ["GraphRAG", "multi-hop reasoning", "relational connectivity", "global dataset summarization", "entity extraction bottleneck", "graph construction cost"]
  },
  {
    "id": "rag_tool_001",
    "category": "tooling_and_frameworks",
    "question": "In a production RAG application serving 50,000 active users, the vector database index needs to be updated hourly as new documents arrive. How do you implement hot-swapping of vector indices without taking retrieval offline or experiencing latency spikes?",
    "expected_concepts": ["hot-swapping", "blue-green indexing", "memory-mapped files", "shadow index / read replica", "atomic pointer swap", "zero downtime"]
  }
]

with open("rag_eval.jsonl", "w", encoding="utf-8") as f:
    for item in RAG_EVAL_BENCHMARK:
        f.write(json.dumps(item) + "\n")

print(f"✅ rag_eval.jsonl auto-initialized with {len(RAG_EVAL_BENCHMARK)} canonical test questions!")

### Step 4: System Prompt & Multi-Objective Reward Functions
Upgrades reward functions with **CoT Planning Bonuses** and **Anti-Hallucination Penalties** for URLs and forum chatter.

In [ ]:
import re
from typing import Any, Dict, List, Optional, Set

SYSTEM_PROMPT = (
    "You are an expert RAG systems architect and researcher. "
    "When presented with a technical challenge or question, first reason through the problem step-by-step "
    "inside <think>...</think> tags, considering architectural trade-offs, failure modes, and underlying mechanisms. "
    "Then, provide your definitive, grounded technical recommendation inside <answer>...</answer> tags."
)

RAG_CONCEPT_TAXONOMY = {
    "retrieval": [
        "dense embedding", "bi-encoder", "cross-encoder", "reranker", "bm25",
        "sparse retrieval", "splade", "hybrid search", "reciprocal rank fusion", "rrf",
        "hnsw", "approximate nearest neighbor", "ann", "cosine similarity",
        "colbert", "hyde", "query expansion", "two-stage retrieval"
    ],
    "chunking_and_indexing": [
        "chunk size", "chunk overlap", "sliding window", "recursive character splitter",
        "semantic chunking", "parent document retriever", "sentence window retrieval",
        "hierarchical indexing", "metadata filtering", "vector database", "qdrant", "milvus", "chroma"
    ],
    "generation_and_grounding": [
        "in-context learning", "context stuffing", "lost in the middle", "faithfulness",
        "hallucination mitigation", "citation grounding", "system prompt", "attribution"
    ],
    "evaluation_and_benchmarks": [
        "ragas", "truelens", "deepeval", "context precision", "context recall",
        "faithfulness", "answer relevance", "llm-as-a-judge", "mrr", "ndcg", "hit rate"
    ],
    "agentic_and_graph_rag": [
        "agentic rag", "graph rag", "knowledge graph", "neo4j", "cypher",
        "adaptive routing", "corrective rag", "crag", "self-rag", "tool calling", "multi-hop reasoning"
    ],
    "tooling_and_frameworks": [
        "langchain", "llamaindex", "haystack", "dspy", "ollama", "vllm", "fastapi"
    ]
}
ALL_RAG_CONCEPTS = {t for sub in RAG_CONCEPT_TAXONOMY.values() for t in sub}

def extract_text(completion: Any) -> str:
    """Extract text string whether completion is a str, dict, or list of message dicts."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, dict):
                parts.append(str(item.get("content", "")))
            elif isinstance(item, str):
                parts.append(item)
            else:
                parts.append(str(item))
        return "\n".join(parts)
    if isinstance(completion, dict):
        return str(completion.get("content", ""))
    return str(completion)

def normalize_text(text: Any) -> str:
    text_str = extract_text(text).lower()
    text_str = re.sub(r"[^\w\s-]", " ", text_str)
    return re.sub(r"\s+", " ", text_str).strip()

def reward_reasoning_format(completions: List[Any], **kwargs) -> List[float]:
    rewards = []
    for item in completions:
        text = extract_text(item)
        has_open = "<think>" in text
        has_close = "</think>" in text
        if has_open and has_close:
            match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
            if match and len(match.group(1).strip()) >= 50:
                has_ans_tags = "<answer>" in text and "</answer>" in text
                rewards.append(1.0 if has_ans_tags else 0.8)
            else:
                rewards.append(0.4)
        elif has_open or has_close:
            rewards.append(-0.5)
        else:
            rewards.append(-1.0)
    return rewards

def reward_rag_concept_density(completions: List[Any], taxonomy_topic: Optional[List[str]] = None, **kwargs) -> List[float]:
    rewards = []
    for idx, item in enumerate(completions):
        text = extract_text(item)
        norm_text = normalize_text(text)
        spaced_text = norm_text.replace("-", " ")
        score = 0.0
        matched = set()
        cat = taxonomy_topic[idx] if taxonomy_topic and idx < len(taxonomy_topic) else ""
        category_vocab = RAG_CONCEPT_TAXONOMY.get(cat, [])

        # CoT Planning Bonus: Award points for concepts brainstormed inside <think>
        think_match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
        if think_match:
            think_norm = normalize_text(think_match.group(1))
            think_spaced = think_norm.replace("-", " ")
            for term in (category_vocab if cat else ALL_RAG_CONCEPTS):
                term_clean = normalize_text(term)
                if term_clean in think_norm or term_clean in think_spaced:
                    score += 0.15

        # Category-specific mechanisms in final completion
        for term in category_vocab:
            term_clean = normalize_text(term)
            if (term_clean in norm_text or term_clean in spaced_text) and term_clean not in matched:
                matched.add(term_clean)
                score += 0.4

        # General RAG concepts
        for term in ALL_RAG_CONCEPTS:
            term_clean = normalize_text(term)
            if (term_clean in norm_text or term_clean in spaced_text) and term_clean not in matched:
                matched.add(term_clean)
                score += 0.2

        rewards.append(min(round(score, 3), 2.0))
    return rewards

def reward_anti_repetition(completions: List[Any], **kwargs) -> List[float]:
    rewards = []
    for item in completions:
        text = extract_text(item)
        if len(text.strip()) < 30:
            rewards.append(-1.5)
            continue
        penalty = 0.0
        words = text.lower().split()
        if len(words) >= 16:
            four_grams = [tuple(words[i:i+4]) for i in range(len(words)-3)]
            unique_ratio = len(set(four_grams)) / len(four_grams)
            if unique_ratio < 0.65:
                penalty -= (0.65 - unique_ratio) * 2.0

        # Penalize hallucinated URLs (-0.4)
        if re.search(r"https?://|www\.", text, flags=re.IGNORECASE):
            penalty -= 0.4

        # Penalize forum chatter / sign-offs (-0.3)
        if re.search(r"\b(hope (this|it) helps|good luck|cheers|let me know if you have questions)\b", text, flags=re.IGNORECASE):
            penalty -= 0.3

        rewards.append(round(penalty, 3))
    return rewards

print("✅ Enhanced reward functions configured successfully!")

## 🌟 Stage 1: Cold-Start SFT (Domain Knowledge Acquisition & Dynamic CoT)

**Key Upgrades in this Stage:**
1. **Substantive Answer Filtering:** Excludes 1-line forum brush-offs, training only on verified solutions with `>= 150` characters.
2. **Answer Hygiene:** Automatically scrubs hallucinated URLs and casual forum chatter.
3. **Dynamic Chain-of-Thought Synthesis:** Generates diverse, content-aware `<think>` traces analyzing the exact problem, failure mode, and architecture instead of a repetitive 3-line boilerplate.

In [ ]:
# 1. Bypass torchao version check if present in Colab runtime
import peft.import_utils as p_utils
p_utils.is_torchao_available = lambda: False
try:
    import peft.utils.quantization_utils as q_utils
    q_utils.is_torchao_available = lambda: False
except Exception:
    pass

import re
import polars as pl
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_OUTPUT_DIR = "sft_rag_adapter"

# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Helpers for Dynamic Chain-of-Thought & Answer Hygiene
FAILURE_MODES = {
    'retrieval': 'lexical mismatch, out-of-vocabulary representation collapse, or reranking latency overhead',
    'chunking_and_indexing': 'embedding dilution from oversized chunks, lost boundary context, or index rebuild cost',
    'generation_and_grounding': 'attention degradation in long contexts (lost-in-the-middle), phantom citations, or hallucination drift',
    'evaluation_and_benchmarks': 'verbosity bias in LLM judges, metric conflation between relevance and faithfulness, or uncalibrated scoring',
    'agentic_and_graph_rag': 'infinite routing loops, relational graph extraction bottlenecks, or state drift across multi-hop reasoning',
    'tooling_and_frameworks': 'framework abstraction lock-in, serialization overhead, or cold-start latency spikes'
}

def clean_answer(text: str) -> str:
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)  # Keep text from markdown links
    text = re.sub(r'https?://\S+', '', text)                # Strip raw URLs
    text = re.sub(r'(?i)\b(hope (this|it) helps|cheers|good luck|let me know if you have questions)\b.*', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def build_dynamic_think(title: str, topic: str, ref_ans: str) -> str:
    norm_text = (title + ' ' + ref_ans).lower()
    norm_text = re.sub(r'[^\w\s-]', ' ', norm_text)
    matched = [c for c in ALL_RAG_CONCEPTS if c in norm_text]
    primary_failure = FAILURE_MODES.get(topic, 'architectural trade-offs and performance bottlenecks')
    steps = [
        f'1. Problem Analysis: The question asks about {topic} regarding "{title}".',
        f'2. Diagnostic Mechanism: Key vulnerability to consider is {primary_failure}.'
    ]
    if matched:
        top_concepts = matched[:3]
        steps.append(f'3. Target Architectural Components: Evaluated mechanisms include {", ".join(top_concepts)}.')
    else:
        steps.append(f'3. Solution Strategy: Formulate grounded engineering principles for {topic}.')
    steps.append('4. Verified Resolution: Structure definitive technical guidance with explicit operational steps.')
    return '<think>\n' + '\n'.join(steps) + '\n</think>'

# 4. Format Dataset with High-Quality Filter and Dynamic CoT
df = pl.read_parquet("grpo_train_dataset.parquet")
print(f"Loaded {len(df)} community records for SFT.")

sft_records = []
# Filter for substantive answers (len >= 150 chars) and build dynamic CoT
filtered_df = df.filter(pl.col("reference_answer").str.len_chars() >= 150)
print(f"Filtered {len(filtered_df)} high-depth technical solutions (>=150 chars).")

for row in filtered_df.head(1500).iter_rows(named=True):
    title = row.get("title") or ""
    topic = row.get("taxonomy_topic") or "retrieval"
    raw_ans = row.get("reference_answer") or ""
    prompt_msgs = row.get("prompt")
    if not prompt_msgs:
        continue

    cleaned_ans = clean_answer(raw_ans)
    if len(cleaned_ans) < 100:
        continue

    user_question = prompt_msgs[-1]["content"]
    dynamic_think = build_dynamic_think(title, topic, cleaned_ans)
    reasoned_completion = f"{dynamic_think}\n<answer>\n{cleaned_ans}\n</answer>"

    full_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_question},
        {"role": "assistant", "content": reasoned_completion}
    ]
    formatted_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    sft_records.append({"text": formatted_text})

sft_dataset = Dataset.from_pandas(pl.DataFrame(sft_records).to_pandas())
print(f"Prepared {len(sft_dataset)} dynamic CoT SFT training examples.")

# 5. LoRA PEFT Configuration
sft_peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# 6. Training Arguments with SFTConfig (T4 memory-safe)
sft_args = SFTConfig(
    output_dir=SFT_OUTPUT_DIR,
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch size = 8
    learning_rate=2e-4,
    num_train_epochs=1,
    max_steps=200,
    logging_steps=20,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=MODEL_ID,
    args=sft_args,
    train_dataset=sft_dataset,
    peft_config=sft_peft_config,
)

print("🚀 Starting Stage 1: Cold-Start SFT (Dynamic CoT + Domain Acquisition)...")
sft_trainer.train()
sft_trainer.save_model(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)
print(f"✅ Stage 1 SFT Adapter saved to: {SFT_OUTPUT_DIR}")

### Step 5: Merge SFT Adapter for Stage 2 Initialization
Merges the SFT adapter weights into the base model so Stage 2 GRPO starts from a model that already speaks the domain vocabulary and reasons inside `<think>`.

In [ ]:
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

del sft_trainer
torch.cuda.empty_cache()
gc.collect()

SFT_MERGED_DIR = "sft_rag_merged"
print("Merging Stage 1 SFT LoRA weights for GRPO initialization...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
merged_sft_model = PeftModel.from_pretrained(base_model, SFT_OUTPUT_DIR)
merged_sft_model = merged_sft_model.merge_and_unload()
merged_sft_model.save_pretrained(SFT_MERGED_DIR)
tokenizer.save_pretrained(SFT_MERGED_DIR)

del base_model, merged_sft_model
torch.cuda.empty_cache()
gc.collect()
print(f"✅ Stage 1 SFT model merged and ready at: {SFT_MERGED_DIR}")

## 🎯 Stage 2: GRPO Reinforcement Learning (Reasoning Policy Refinement)

Trains GRPO on top of the SFT model using the upgraded multi-objective reward functions with **CoT Planning Bonuses**.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

GRPO_OUTPUT_DIR = "grpo_rag_adapter"

# Load RL dataset (use 1,000 prompts for Colab session)
rl_dataset = Dataset.from_pandas(df.head(1000).to_pandas())

# LoRA adapter configuration for GRPO
grpo_peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# GRPO configuration (T4 GPU memory-safe optimized with 320 completion headroom)
grpo_args = GRPOConfig(
    output_dir=GRPO_OUTPUT_DIR,
    learning_rate=1.5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,      # frees ~3.5 GB VRAM
    num_generations=4,                  # Group size G=4
    max_completion_length=320,          # Headroom for think + full answer
    max_steps=100,
    logging_steps=10,
    save_steps=25,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
)

# Train GRPO on top of the SFT-prepared model!
grpo_trainer = GRPOTrainer(
    model=SFT_MERGED_DIR,
    reward_funcs=[reward_reasoning_format, reward_rag_concept_density, reward_anti_repetition],
    args=grpo_args,
    train_dataset=rl_dataset,
    peft_config=grpo_peft_config,
)

print("🚀 Starting Stage 2: GRPO Reinforcement Learning (Reasoning Policy Refinement)...")
grpo_trainer.train()
grpo_trainer.save_model(GRPO_OUTPUT_DIR)
tokenizer.save_pretrained(GRPO_OUTPUT_DIR)
print(f"✅ Stage 2 GRPO LoRA adapter saved to: {GRPO_OUTPUT_DIR}")

## 🔬 Stage 3: Deep Diagnostic Evaluation on Held-Out Benchmark & Full Debugging Report

Evaluates on the 13 canonical held-out test cases (`rag_eval.jsonl`) with `max_new_tokens=512` so answers are never truncated.

In [ ]:
import os
import gc
import re
import json
import time
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Clean GPU memory
torch.cuda.empty_cache()
gc.collect()

print("=" * 70)
print("🎯 RUNNING POST-ALIGNMENT EVALUATION & DEBUGGING HARNESS")
print("=" * 70)

# Load model with GRPO LoRA weights
eval_tokenizer = AutoTokenizer.from_pretrained(SFT_MERGED_DIR, trust_remote_code=True)
eval_base = AutoModelForCausalLM.from_pretrained(
    SFT_MERGED_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
eval_model = PeftModel.from_pretrained(eval_base, GRPO_OUTPUT_DIR)
eval_model.eval()

# Load held-out benchmark questions
with open("rag_eval.jsonl", "r", encoding="utf-8") as f:
    eval_questions = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(eval_questions)} held-out benchmark questions.\n")

# Helper functions for robust concept matching
def norm_t(t: str) -> str:
    t = t.lower()
    t = re.sub(r"[^\w\s-]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def get_variants(concept: str) -> set:
    variants = set()
    parts = re.split(r"\s*/\s*|\s+or\s+", concept)
    for p in parts:
        clean = norm_t(p)
        if clean:
            variants.add(clean)
        parens = re.findall(r"\(([^)]+)\)", p)
        for pr in parens:
            pr_c = norm_t(pr)
            if pr_c:
                variants.add(pr_c)
        no_pr = norm_t(re.sub(r"\([^)]*\)", "", p))
        if no_pr:
            variants.add(no_pr)
    hyphen_v = set()
    for v in variants:
        if "-" in v:
            hyphen_v.add(v.replace("-", " "))
            hyphen_v.add(v.replace("-", ""))
    variants.update(hyphen_v)
    return {v for v in variants if len(v) >= 2}

results = []
category_recalls = {}

for idx, item in enumerate(eval_questions, 1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["question"]}
    ]
    prompt_str = eval_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = eval_tokenizer(prompt_str, return_tensors="pt").to("cuda")
    
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=512,        # Adequate token budget for think + complete answer
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            pad_token_id=eval_tokenizer.pad_token_id or eval_tokenizer.eos_token_id
        )
    elapsed = time.perf_counter() - t0
    
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    tok_count = len(new_tokens)
    tok_per_sec = round(tok_count / max(elapsed, 0.001), 2)
    decoded = eval_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Extract thinking trace and final answer
    has_think = ("<think>" in decoded and "</think>" in decoded)
    think_trace = ""
    final_answer = decoded
    if has_think:
        m = re.search(r"<think>(.*?)</think>", decoded, flags=re.DOTALL)
        if m:
            think_trace = m.group(1).strip()
        ans_m = re.search(r"<answer>(.*?)</answer>", decoded, flags=re.DOTALL)
        if ans_m:
            final_answer = ans_m.group(1).strip()
        elif "</think>" in decoded:
            final_answer = decoded.split("</think>")[-1].strip()
            
    # Concept coverage computation
    norm_resp = norm_t(decoded)
    spaced_resp = norm_resp.replace("-", " ")
    matched, missing = [], []
    
    for concept in item["expected_concepts"]:
        variants = get_variants(concept)
        if any((f" {v} " in f" {norm_resp} ") or (f" {v} " in f" {spaced_resp} ") for v in variants):
            matched.append(concept)
        else:
            missing.append(concept)
            
    recall = round(len(matched) / max(len(item["expected_concepts"]), 1), 4)
    cat = item["category"]
    category_recalls.setdefault(cat, []).append(recall)
    
    q_result = {
        "id": item["id"],
        "category": cat,
        "question": item["question"],
        "recall": recall,
        "tokens": tok_count,
        "tok_per_sec": tok_per_sec,
        "has_think": has_think,
        "thinking_trace": think_trace,
        "final_answer": final_answer,
        "raw_completion": decoded,
        "matched_concepts": matched,
        "missing_concepts": missing
    }
    results.append(q_result)
    
    status = "🧠 [THINKING]" if has_think else "📝 [DIRECT]"
    print(f"[{idx:02d}/{len(eval_questions):02d}] {item['id']:<14} | Recall: {recall*100:>5.1f}% | {status} | Speed: {tok_per_sec:>5.1f} t/s")
    print(f"     ✅ Matched: {matched}")
    if missing:
        print(f"     ❌ Missing (Fumbled): {missing}")
    print("-" * 70)

# Summary Metrics
avg_recall = round(sum(r["recall"] for r in results) / len(results) * 100, 2)
think_pct = round(sum(1 for r in results if r["has_think"]) / len(results) * 100, 2)
cat_summary = {cat: round(sum(scores) / len(scores) * 100, 1) for cat, scores in category_recalls.items()}

report = {
    "model_id": "RAGForge-Qwen2.5-1.5B-TwoStage-SFT-GRPO",
    "baseline_recall_pct": 23.08,
    "post_trained_recall_pct": avg_recall,
    "delta_improvement_pct": round(avg_recall - 23.08, 2),
    "thinking_trace_pct": think_pct,
    "category_summary": cat_summary,
    "results": results
}

with open("post_grpo_eval_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 70)
print(f"🏆 EVALUATION COMPLETE!")
print(f"   Vanilla Baseline Recall: 23.08%")
print(f"   Trained Model Recall:    {avg_recall}% (Δ {report['delta_improvement_pct']:+0.2f}%)")
print(f"   Thinking Trace Adoption: {think_pct}%")
print(f"   Full Debugging Log:      Saved to post_grpo_eval_report.json (includes all prompt/thinking/answer/fumble logs)")
print("=" * 70)

## 📤 Step 7: Push to Hugging Face Hub (LoRA Adapter & Standalone Model)
Uploads the trained LoRA adapter (~74 MB) and the merged standalone model (~3.1 GB) directly to Hugging Face Hub.

In [ ]:
from huggingface_hub import login
from google.colab import userdata
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Authenticate with Hugging Face
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception:
    print("Please login manually with your Hugging Face write token:")
    login()

HF_USERNAME = "YashJO"  # Replace with your Hugging Face username
ADAPTER_REPO = f"{HF_USERNAME}/RAGForge-Qwen2.5-1.5B-GRPO-Adapter"
STANDALONE_REPO = f"{HF_USERNAME}/RAGForge-Qwen2.5-1.5B-GRPO"

# 2. Push LoRA Adapter (~74 MB)
print(f"Uploading LoRA Adapter to: {ADAPTER_REPO}")
eval_model.push_to_hub(ADAPTER_REPO, private=False)
eval_tokenizer.push_to_hub(ADAPTER_REPO, private=False)
print(f"✅ LoRA Adapter uploaded: https://huggingface.co/{ADAPTER_REPO}")

# 3. Merge and Push Standalone Model (~3.1 GB)
print(f"\nMerging weights and uploading Standalone Model to: {STANDALONE_REPO}...")
merged_final = eval_model.merge_and_unload()
merged_final.push_to_hub(STANDALONE_REPO, private=False)
eval_tokenizer.push_to_hub(STANDALONE_REPO, private=False)
print(f"✅ Standalone Model uploaded: https://huggingface.co/{STANDALONE_REPO}")

## 💾 Step 8: Download Local Artifacts Backup
Zips and downloads the adapter weights and the full `post_grpo_eval_report.json` to your local machine.

In [ ]:
!zip -r ragforge_artifacts.zip grpo_rag_adapter post_grpo_eval_report.json
from google.colab import files
files.download("ragforge_artifacts.zip")
print("✅ Downloaded adapter and eval debug report successfully!")